# Transaltion Tables
- mapping structures that define how to replace or delete specific characters in a string

## What They Actually Are
At their core, translation tables are dictionaries mapping Unicode code points (integers) to:
- Another Unicode code point (replace character)
- `None` (delete character)
- `None` (leave unchanged - implied if not in table)

```python
# A translation table as a dictionary
{
    97: 49,    # 'a' → '1'
    98: None,  # 'b' → delete
    99: 99     # 'c' → unchanged (not needed)
}
```

## Basic Character Replacement

In [ ]:
# Create translation table
trans_table = str.maketrans(
    "aeiou",     # characters to replace
    "12345"      # replacement characters (same length)
)

text = "hello world"
print(text.translate(trans_table))
# Output: h2ll4 w4rld

## Character Deletion

In [ ]:
# Delete specific characters
trans_table = str.maketrans("", "", "!@#$%^&*()")

text = "Hello! World@ Python$"
print(text.translate(trans_table))
# Output: Hello World Python

## Using Dictionary for Complex Mappings

In [ ]:
# Dictionary format: {ordinal: replacement}
trans_table = str.maketrans({
    'a': '4',
    'e': '3', 
    'i': '1',
    'o': '0',
    's': '5'
})

text = "hello world"
print(text.translate(trans_table))
# Output: h3ll0 w0rld

# Real-World Examples

## Text Cleaner

In [ ]:
def clean_text(text):
    """Remove punctuation and convert to lowercase"""
    trans_table = str.maketrans(
        "", "", 
        ".,!?;:()[]{}'\"\\/|`~@#$%^&*_-+=<>"
    )
    return text.translate(trans_table).lower()

print(clean_text("Hello, World! How are you?"))
# Output: hello world how are you

## Password Geenrator Helper

In [ ]:
def leetspeak(text):
    """Convert text to leetspeak"""
    leet_map = {
        'a': '4', 'e': '3', 'i': '1', 'o': '0',
        's': '5', 't': '7', 'b': '8', 'g': '9'
    }
    trans_table = str.maketrans(leet_map)
    return text.translate(trans_table)

print(leetspeak("python is great"))
# Output: py7h0n 1s 9r347

## Unicode Normalizaiton

In [ ]:
import unicodedata
def remove_accents(text):
    """Remove diacritical marks from text"""
    
    # Normalize to decomposed form
    normalized = unicodedata.normalize('NFD', text)
    
    # Remove combining diacritical marks
    trans_table = str.maketrans("", "", 
        ''.join(chr(i) for i in range(0x0300, 0x0370)))
    
    return normalized.translate(trans_table)

print(remove_accents("café naïve résumé"))
# Output: cafe naive resume

# Custom Translation Class

In [ ]:
class TranslationTable:
    def __init__(self):
        self.mappings = {}
    
    def add_mapping(self, char_from, char_to):
        self.mappings[ord(char_from)] = char_to
        return self
    
    def add_deletion(self, *chars):
        for char in chars:
            self.mappings[ord(char)] = None
        return self
    
    def build(self):
        return str.maketrans(self.mappings)
    
    def translate(self, text):
        return text.translate(self.build())

# Usage
translator = (TranslationTable()
    .add_mapping('a', '4')
    .add_mapping('e', '3')
    .add_deletion('!', '@', '#')
    .translate)

text = "hello @world! python"
print(translator(text))
# Output: h3llo world python

# Performance Comparison

In [ ]:
import time

def test_translation_methods():
    text = "hello world " * 10000
    
    # Method 1: translate()
    trans_table = str.maketrans("aeiou", "12345")
    
    start = time.time()
    result1 = text.translate(trans_table)
    time1 = time.time() - start
    
    # Method 2: replace() - slower for multiple replacements
    start = time.time()
    result2 = text
    for old, new in [('a','1'), ('e','2'), ('i','3'), ('o','4'), ('u','5')]:
        result2 = result2.replace(old, new)
    time2 = time.time() - start
    
    print(f"translate(): {time1:.4f}s")
    print(f"replace(): {time2:.4f}s")
    print(f"translate is {time2/time1:.1f}x faster")

test_translation_methods()

# Advanced: Bidirectional Translation

In [ ]:
class BidirectionalTranslator:
    def __init__(self, forward_map, backward_map=None):
        self.forward = str.maketrans(forward_map)
        self.backward = str.maketrans(backward_map or 
                                      {v: k for k, v in forward_map.items()})
    
    def encode(self, text):
        return text.translate(self.forward)
    
    def decode(self, text):
        return text.translate(self.backward)

# Caesar cipher example
alphabet = 'abcdefghijklmnopqrstuvwxyz'
shift = 3
caesar_map = {alphabet[i]: alphabet[(i+shift) % 26] for i in range(26)}

cipher = BidirectionalTranslator(caesar_map)

original = "hello world"
encoded = cipher.encode(original)
decoded = cipher.decode(encoded)

print(f"Original: {original}")
print(f"Encoded: {encoded}")
print(f"Decoded: {decoded}")

# Why "Translation Tables"?
The name comes from character translation - converting one set of characters to another, like a lookup table or a "code book" for character conversion. It's similar to:
- Caesar cipher mapping tables
- ASCII/Unicode code pages
- Character encoding conversion tables
- Language transliteration tables

When to Use Translation Tables
✅ Perfect for:
- Removing punctuation
- Converting case (though .lower() is simpler for that)
- Simple character substitutions (leetspeak, ciphers)
- Normalizing whitespace characters
- Filtering out special characters

❌ Not for:
- Replacing substrings ("hello" → "goodbye")
- Pattern-based replacements (use regex)
- Complex conditional logic